# Memory Mosaics in JAX: Fig. 7, single block

Trains a 1-block GPT-2 and a 1-block Memory Mosaic on **BabiStories** and plots training and validation loss,
as in the 1-block panel of Fig. 7 (arXiv:2405.06394v3).

**Quick run (default, ~15 min on a T4).** The paper's model (d=768, 12 heads, context 512), data, optimizer
and dropout, but batches of 32 sequences instead of 512 and a schedule sized to a 10-minute training budget.
That covers roughly 5–10M tokens, the very start of training. The paper's figure starts after 2,000 iterations
of 512 sequences (about 1B tokens) and runs to 80,000, so the quick run shows a different stretch of the curve
and is not comparable to it point for point.

**Paper run.** `PAPER_RUN = True` uses the paper's full recipe: 80,000 steps of 512 sequences, about 21B
tokens per model. That needs a long-lived A100-class machine, not a Colab session.

Runtime → Change runtime type → **T4 GPU**, then run top to bottom.

In [ ]:
# Colab already ships JAX with CUDA; reinstalling jax[cuda12] registers the GPU plugin twice.
!pip install -q optax tiktoken py7zr multivolumefile matplotlib
import os, sys
if not os.path.exists("memory-mosaics-jax"):
    !git clone -q https://github.com/rohitium/memory-mosaics-jax.git
sys.path.insert(0, "memory-mosaics-jax/src")

import jax
print(jax.devices())

## Models

Both use the paper's sizes. A Mosaic block has about the same parameter count as a GPT-2 block.

In [ ]:
from mm_jax import config, gpt2, memory_mosaic, train
from mm_jax.nn import count_params

PAPER_RUN = False
TRAIN_MINUTES = 10  # quick run: wall-clock training budget for both models together

g_cfg = config.ModelConfig(n_layer=1)
m_cfg = config.MosaicConfig(n_layer=1)
for name, module, cfg in [("GPT-2", gpt2, g_cfg), ("Mosaic", memory_mosaic, m_cfg)]:
    p = module.init(jax.random.PRNGKey(0), cfg)
    print(f"{name:6s} block params: {count_params(p['blocks']):,}   total: {count_params(p):,}")

## Data: BabiStories

Downloads the dataset from the paper's repo (~470 MB) and tokenizes it with GPT-2 BPE exactly as the reference
`prepare.py` does. The quick run only needs the first ~20M training tokens; validation always uses the full
4.7M-token split.

In [ ]:
import numpy as np
from mm_jax.data import TextBatcher, prepare_babistories

bins = prepare_babistories("/content/babistories", max_train_tokens=None if PAPER_RUN else 20_000_000)
train_ids, val_ids = (np.memmap(bins[s], dtype=np.uint16, mode="r") for s in ("train", "val"))
print(f"{len(train_ids):,} train / {len(val_ids):,} val tokens")

## Training schedule

The quick run times a few steps of each model on this GPU, then sets the number of steps so both models fit
the budget. Warmup is 5% of the steps, and the cosine decay to 1e-4 spans the whole run, as in the paper.

In [ ]:
import time

def seconds_per_step(module, cfg, t_cfg):
    train_step, _, state = train.make_trainer(module.apply, module.init(jax.random.PRNGKey(0), cfg), cfg, t_cfg)
    batch = TextBatcher(train_ids).next_batch(t_cfg.batch_size, cfg.block_size)
    state, loss = train_step(state, batch, jax.random.PRNGKey(0))  # compile
    loss.block_until_ready()
    t0 = time.time()
    for i in range(3):
        state, loss = train_step(state, batch, jax.random.PRNGKey(i))
    loss.block_until_ready()
    return (time.time() - t0) / 3

if PAPER_RUN:
    t_cfg = config.TrainConfig()
else:
    quick = dict(batch_size=32, micro_batch_size=8)
    probe = config.TrainConfig(**quick, max_steps=100, warmup_steps=5)
    per_step = seconds_per_step(gpt2, g_cfg, probe) + seconds_per_step(memory_mosaic, m_cfg, probe)
    steps = int(TRAIN_MINUTES * 60 / (1.1 * per_step))  # ~10% for evaluation
    t_cfg = config.TrainConfig(**quick, max_steps=steps, warmup_steps=max(1, steps // 20),
                               eval_every=max(1, steps // 8), eval_batches=8)
tokens = t_cfg.max_steps * t_cfg.batch_size * g_cfg.block_size
print(f"{t_cfg.max_steps:,} steps x {t_cfg.batch_size} sequences = {tokens / 1e6:,.1f}M tokens per model")

## Train both models

Same data stream, optimizer and schedule for both models.

In [ ]:
def run(name, module, cfg):
    train_step, eval_step, state = train.make_trainer(module.apply, module.init(jax.random.PRNGKey(0), cfg), cfg, t_cfg)
    train_b, val_b, key = TextBatcher(train_ids, seed=0), TextBatcher(val_ids, seed=1), jax.random.PRNGKey(1)
    log = {"train": [], "val_steps": [], "val": []}
    t0 = time.time()
    for step in range(1, t_cfg.max_steps + 1):
        key, k = jax.random.split(key)
        state, loss = train_step(state, train_b.next_batch(t_cfg.batch_size, cfg.block_size), k)
        log["train"].append(float(loss))
        if step % t_cfg.eval_every == 0 or step == t_cfg.max_steps:
            log["val_steps"].append(step)
            log["val"].append(train.eval_loss(eval_step, state["params"], val_b, t_cfg, cfg.block_size))
            print(f"[{name}] step {step:6d}  train {log['train'][-1]:.3f}  val {log['val'][-1]:.3f}  ({time.time() - t0:.0f}s)")
    return log

g_log = run("gpt2", gpt2, g_cfg)
m_log = run("mosaic", memory_mosaic, m_cfg)

## Plot

In [ ]:
import matplotlib.pyplot as plt

W = max(1, t_cfg.max_steps // 20)  # smoothing window for the per-step training loss
plt.figure(figsize=(8, 5))
for log, label, c in [(g_log, "GPT-2", "C0"), (m_log, "Memory Mosaic", "C1")]:
    smooth = np.convolve(log["train"], np.ones(W) / W, mode="valid")
    plt.plot(np.arange(W, len(log["train"]) + 1), smooth, color=c, lw=1.5, label=f"{label}, train")
    plt.plot(log["val_steps"], log["val"], "o--", color=c, ms=4, label=f"{label}, val")
plt.xlabel(f"training iterations (batch {t_cfg.batch_size} x {g_cfg.block_size} tokens)")
plt.ylabel("cross-entropy loss")
plt.title(f"1 block: Memory Mosaic vs GPT-2 on BabiStories ({tokens / 1e6:,.0f}M tokens)")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("fig7_1block.png", dpi=150, bbox_inches="tight")
plt.show()